# Objective of this script

Currently the data is organized such that each day we have one file for each city, which contains the forecasts for the next X days. It would be much better to have one large file containing the full data and meta data:

**Meta data**
- Location name
- Coordinates
- Date & time when forecast was made
- Date & time for which the forecast is made

**Data**
- Measurable weather data

Import Python packages

In [1]:
import pandas as pd
import numpy as np
import os
import json

Create the target data file that will contain the data in the new format

In [2]:
target_data_file_name = "forecasting_data.csv"
forecasting_data = pd.DataFrame()

The city location and weather metrics are stored in the user config file.

In [3]:
CONFIG_FILE = "../data/user/.fue_config.json"
with open(CONFIG_FILE) as configfile:
    config_dict = json.load(configfile)
cities = config_dict["cities"]
metrics = config_dict["metrics"]

Update columns of target data file

In [4]:
columns = [
    "location name",
    "latitude",
    "longitude",
    "forecasted_on",
    "forecast_for"]
# Now add physical weather metrics to the column list
columns = columns + metrics

Now we iterate through each file and in each file through each forecast and append the forecast to the forecasting_data DataFrame.

In [5]:
RAW_DATA = "../data/raw/"
PROCESSED_DATA = "../data/processed"
files = os.listdir(RAW_DATA)
for file in files:

    # Read Data
    file_path = RAW_DATA + file
    df = pd.read_csv(file_path)

    # Extract meta data
    city_name, date_str = file[:-4].split("_") # filename looks like: aachen_20260601.csv
    lat = cities[city_name]["lat"]
    lon = cities[city_name]["lon"]
    forecasted_on = pd.to_datetime(f"{date_str}12", format="%Y%m%d%H") # assume that previous forecasts were downloaded roughly around midday

    # Create new 1-row DataFrame, which we can then append to the global DataFrame.
    new_row = {}
    new_row["location_name"] = city_name
    new_row["latitude"] = lat
    new_row["longitude"] = lon
    new_row["forecasted_on"] = forecasted_on

    for row in df.iterrows():
        data_point = row[1] # First item is just an index, the second item contains the data
        new_row["forecast_for"] = pd.to_datetime(f"{data_point["date"]}23:59:59", format="%Y-%m-%d%H:%M:%S") # assume that previous forecasts were downloaded roughly around midday
        for metric in metrics:
            try:
                new_row[metric] = data_point[metric]
            except:
                new_row[metric] = np.nan
        forecasting_data = pd.concat([forecasting_data, pd.DataFrame([new_row])], ignore_index=True)

In [6]:
forecasting_data.sort_values(["location_name", "forecasted_on"])

,location_name,latitude,longitude,forecasted_on,forecast_for,temperature_2m_max,temperature_2m_min,precipitation_sum,sunshine_duration,wind_direction_10m_dominant,wind_speed_10m_max,wind_gusts_10m_max,wind_speed_10m_mean,precipitation_probability_mean
2652,aachen,50.7753,6.0839,2026-04-15 12:00:00,2026-04-15 23:59:59,19.126,7.826,0.0,44919.938,197.13081,12.240000,24.119999,8.340000,1.041667
2653,aachen,50.7753,6.0839,2026-04-15 12:00:00,2026-04-16 23:59:59,18.526,11.926,0.1,35147.580,247.85912,12.959999,25.919998,8.400000,10.791667
2654,aachen,50.7753,6.0839,2026-04-15 12:00:00,2026-04-17 23:59:59,20.376,12.776,0.0,42178.040,262.14413,6.840000,19.080000,4.050000,0.000000
2655,aachen,50.7753,6.0839,2026-04-15 12:00:00,2026-04-18 23:59:59,17.950,10.000,7.1,41540.830,247.76485,10.742848,27.359999,6.400315,22.500000
2656,aachen,50.7753,6.0839,2026-04-15 12:00:00,2026-04-19 23:59:59,13.900,7.800,4.2,48234.727,338.25090,15.188417,37.440000,11.015160,35.125000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
676,zurich,47.3769,8.5472,2026-06-16 12:00:00,2026-06-25 23:59:59,30.800,20.850,0.0,55342.508,50.80102,9.761578,28.800000,6.070393,11.791667
677,zurich,47.3769,8.5472,2026-06-16 12:00:00,2026-06-26 23:59:59,32.500,21.150,0.0,56585.150,56.16764,14.642090,38.880000,9.113033,8.166667
678,zurich,47.3769,8.5472,2026-06-16 12:00:00,2026-06-27 23:59:59,33.400,22.000,0.0,55987.426,191.94383,5.545052,30.239998,2.453790,3.583333
679,zurich,47.3769,8.5472,2026-06-16 12:00:00,2026-06-28 23:59:59,34.150,23.500,0.0,56193.617,202.80605,10.609316,30.599998,6.162607,6.625000


Let's see how many ground truth values we have and how many predictions

In [17]:
labels = 0
trainings = 0
for row in forecasting_data.iterrows():
    data = row[1]
    f_on = data["forecasted_on"]
    f_for = data["forecast_for"]
    if (f_on + pd.Timedelta(hours=12) < f_for): trainings += 1
    if (f_on + pd.Timedelta(hours=12) >= f_for): labels += 1
print(trainings, labels)

2056 657


Store the compact data frame as a csv

In [25]:
FILE_NAME = "compact_forecast_data.csv"
FILE_PATH = "../data/processed/"

# check if there is already a compact data file and if so, join them without duplicates
PATH = os.path.join(FILE_PATH, FILE_NAME)
if os.path.exists(PATH):
    existing_data = pd.read_csv(PATH, parse_dates=['forecasted_on', 'forecast_for'])
    existing_count = len(existing_data)
    combined_data = pd.concat([existing_data, forecasting_data], ignore_index=True)
    combined_data = combined_data.drop_duplicates(
        subset=['location_name', 'forecasted_on', 'forecast_for'], 
        keep='last'
    )
    entries_added = len(combined_data) - existing_count
    print(f"Added {entries_added} new entries to the dataset")
    combined_data.to_csv(PATH, index=False)
else:
    forecasting_data.to_csv(PATH, index=False)

Added 0 new entries to the dataset
